# SQL Analysis

## Project Goal

This notebook demonstrates how SQL can be used to analyze e-commerce user behavior, product performance, funnel metrics, and purchase activity.

SQLite is used as a lightweight database engine directly inside Google Colab.

In [10]:
import pandas as pd
import sqlite3

In [11]:
events = pd.read_csv("events.csv")
products = pd.read_csv("products.csv")
users = pd.read_csv("users.csv")
sessions = pd.read_csv("sessions.csv")

## Create SQLite Database

In [12]:
conn = sqlite3.connect("ecommerce.db")

events.to_sql("events", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
users.to_sql("users", conn, if_exists="replace", index=False)
sessions.to_sql("sessions", conn, if_exists="replace", index=False)

print("Database successfully created!")

Database successfully created!


In [13]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,events
1,products
2,users
3,sessions


**Basic Statistics**

In [14]:
query = """
SELECT COUNT(*) AS total_events
FROM events;
"""

pd.read_sql(query, conn)

,total_events
0,500000


In [15]:
query = """
SELECT COUNT(DISTINCT user_id) AS unique_users
FROM events;
"""

pd.read_sql(query, conn)

,unique_users
0,138462


In [16]:
query = """
SELECT COUNT(DISTINCT user_session) AS unique_sessions
FROM events;
"""

pd.read_sql(query, conn)

,unique_sessions
0,182502


**Event Analysis**

In [17]:
query = """
SELECT
event_type,
COUNT(*) AS events
FROM events
GROUP BY event_type
ORDER BY events DESC;
"""

pd.read_sql(query, conn)

,event_type,events
0,view,483943
1,purchase,8746
2,cart,7311


In [18]:
query = """
SELECT
DATE(event_time) AS event_date,
COUNT(*) AS events
FROM events
GROUP BY DATE(event_time);
"""

pd.read_sql(query, conn)

,event_date,events
0,2019-11-01,500000


In [19]:
query = """
SELECT
STRFTIME('%H', event_time) AS hour,
COUNT(*) AS events
FROM events
GROUP BY hour
ORDER BY hour;
"""

pd.read_sql(query, conn)

,hour,events
0,00,5147
1,01,6629
2,02,15482
3,03,23386
4,04,29235
5,05,35052
6,06,36263
7,07,38129
8,08,38080
9,09,36421


**Funnel**

In [20]:
query = """
SELECT
event_type,
COUNT(DISTINCT user_id) AS users
FROM events
WHERE event_type IN ('view','cart','purchase')
GROUP BY event_type;
"""

pd.read_sql(query, conn)

,event_type,users
0,cart,5121
1,purchase,7324
2,view,137213


In [21]:
query = """
WITH funnel AS (
SELECT
COUNT(DISTINCT CASE WHEN event_type='view' THEN user_id END) AS viewers,
COUNT(DISTINCT CASE WHEN event_type='cart' THEN user_id END) AS cart_users,
COUNT(DISTINCT CASE WHEN event_type='purchase' THEN user_id END) AS buyers
FROM events
)

SELECT
viewers,
cart_users,
buyers,
ROUND(cart_users*100.0/viewers,2) AS view_to_cart,
ROUND(buyers*100.0/cart_users,2) AS cart_to_purchase,
ROUND(buyers*100.0/viewers,2) AS overall_conversion
FROM funnel;
"""

pd.read_sql(query, conn)

,viewers,cart_users,buyers,view_to_cart,cart_to_purchase,overall_conversion
0,137213,5121,7324,3.73,143.02,5.34


**Product Analysis**

In [22]:
query = """
SELECT
p.category_code,
COUNT(*) AS purchases
FROM events e
JOIN products p
ON e.product_id=p.product_id
WHERE event_type='purchase'
GROUP BY category_code
ORDER BY purchases DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,category_code,purchases
0,electronics.smartphone,4088
1,None,2184
2,electronics.audio.headphone,351
3,electronics.video.tv,264
4,electronics.clocks,181
5,appliances.kitchen.washer,171
6,computers.notebook,161
7,appliances.environment.vacuum,118
8,appliances.kitchen.refrigerators,116
9,computers.desktop,59


In [23]:
query = """
SELECT
p.brand,
COUNT(*) AS purchases
FROM events e
JOIN products p
ON e.product_id=p.product_id
WHERE event_type='purchase'
GROUP BY brand
ORDER BY purchases DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,brand,purchases
0,samsung,2064
1,apple,1769
2,None,639
3,xiaomi,616
4,huawei,241
5,cordiant,187
6,oppo,152
7,lucente,137
8,nokian,101
9,lg,91


**Revenue Analysis**

In [24]:
query = """
SELECT
brand,
ROUND(SUM(price),2) AS revenue
FROM products
GROUP BY brand
ORDER BY revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,brand,revenue
0,None,2247536.67
1,apple,351780.39
2,hp,301163.83
3,samsung,278788.87
4,bosch,212412.53
5,sony,211807.91
6,lenovo,170309.58
7,lg,162997.36
8,asus,137265.35
9,electrolux,108394.17


In [25]:
query = """
SELECT
category_code,
ROUND(AVG(price),2) AS avg_price
FROM products
GROUP BY category_code
ORDER BY avg_price DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,category_code,avg_price
0,electronics.camera.photo,989.74
1,electronics.video.projector,914.05
2,computers.notebook,908.62
3,furniture.living_room.sofa,843.43
4,electronics.video.tv,642.87
5,sport.trainer,594.44
6,sport.bicycle,513.82
7,appliances.environment.air_conditioner,508.17
8,appliances.kitchen.dishwasher,498.99
9,auto.accessories.winch,497.25


**Customer Analysis**

In [26]:
query = """
SELECT
user_id,
COUNT(*) AS purchases
FROM events
WHERE event_type='purchase'
GROUP BY user_id
ORDER BY purchases DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,user_id,purchases
0,564068124,13
1,519369539,12
2,514544847,12
3,551801600,9
4,547217188,9
5,541328903,9
6,536084345,9
7,526933626,9
8,515934670,9
9,512588810,9


In [27]:
query = """
SELECT
ROUND(AVG(events_per_session),2) AS avg_events_per_session
FROM
(
SELECT
user_session,
COUNT(*) AS events_per_session
FROM events
GROUP BY user_session
);
"""

pd.read_sql(query, conn)


,avg_events_per_session
0,2.74


**Window Functions**

In [28]:
query = """
SELECT
brand,
SUM(price) AS revenue,
RANK() OVER(ORDER BY SUM(price) DESC) AS revenue_rank
FROM products
GROUP BY brand
LIMIT 10;
"""

pd.read_sql(query, conn)

,brand,revenue,revenue_rank
0,None,2247536.67,1
1,apple,351780.39,2
2,hp,301163.83,3
3,samsung,278788.87,4
4,bosch,212412.53,5
5,sony,211807.91,6
6,lenovo,170309.58,7
7,lg,162997.36,8
8,asus,137265.35,9
9,electrolux,108394.17,10


In [29]:
query = """
WITH daily AS
(
SELECT
DATE(event_time) AS day,
COUNT(*) AS purchases
FROM events
WHERE event_type='purchase'
GROUP BY DATE(event_time)
)

SELECT
day,
purchases,
LAG(purchases) OVER(ORDER BY day) AS previous_day
FROM daily;
"""

pd.read_sql(query, conn)

,day,purchases,previous_day
0,2019-11-01,8746,None


## SQL Summary

Using SQL, we explored customer behavior, product performance, funnel metrics, revenue distribution, and user activity.

The analysis included:
- Basic statistics
- Event analysis
- Product analysis
- Revenue analysis
- Customer analysis
- Funnel metrics
- Window functions (RANK, LAG)